# Experiment 12: DeBERTa Fine-tuning

## Overview
- Model: DeBERTa-v3 (Transformers + PyTorch)
- Approach: Train with Stratified 5-Fold CV, save per-fold models and metrics, produce averaged test predictions for submission.
- Features:
  - GPU acceleration with mixed precision training
  - Gradient accumulation for larger effective batch sizes
  - Learning rate warmup and decay
  - Early stopping with patience
  - Optional layer freezing strategies
  - Gradient clipping for stability
  - Class weighting for imbalanced data
- Artifacts (per model tracking guide): model files `model_fold_i/`, `metrics.json`, and submission CSV `exp_11_deberta_submission.csv`.

In [ ]:
# Mount Google Drive for Colab
from google.colab import drive
drive.mount('/content/drive')
use_drive = True

Mounted at /content/drive


In [ ]:
# Install required packages
# !pip install transformers accelerate datasets scikit-learn torch tqdm

In [ ]:
# Imports
import os
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DebertaV2Tokenizer, 
    DebertaV2ForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, 
    recall_score, f1_score, log_loss
)
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
# Initialize paths
DATA_PATH = 'data/'
EXPERIMENT_PATH = 'experiments/'
EXPERIMENT_NAME = 'exp_12_deberta'

if use_drive:
    DRIVE_PATH = '/content/drive/MyDrive/Kaggle'
    EXPERIMENT_PATH = os.path.join(DRIVE_PATH, EXPERIMENT_PATH)

EXPERIMENT_DIR = os.path.join(EXPERIMENT_PATH, EXPERIMENT_NAME)
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
os.makedirs(os.path.join(EXPERIMENT_DIR, 'models'), exist_ok=True)
print('Experiment directory:', EXPERIMENT_DIR)

Experiment directory: /content/drive/MyDrive/Kaggle/experiments/exp_6_bilstm


In [ ]:
# Load data
train = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))

print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'Class distribution:\n{train["label"].value_counts()}')

In [ ]:
# Configuration
CONFIG = {
    # Model settings
    'model_name': 'microsoft/deberta-v3-base',  # Can also try 'microsoft/deberta-v3-large' or 'microsoft/deberta-v3-small'
    'max_length': 512, # 256,  # Maximum sequence length
    'num_labels': 2,
    
    # Training settings
    'n_folds': 5,
    'batch_size': 32,  # 16 # Adjust based on GPU memory (DeBERTa may need smaller batch size than RoBERTa)
    'gradient_accumulation_steps': 1, # 2  # Effective batch size = batch_size * gradient_accumulation_steps
    'num_epochs': 4,  # Typically 3-5 epochs for transformer fine-tuning
    'learning_rate': 2e-5,  # Standard for DeBERTa fine-tuning
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,  # 10% of training steps for warmup
    'max_grad_norm': 1.0,  # Gradient clipping
    
    # Mixed precision training
    'use_fp16': True,  # Use mixed precision for faster training
    
    # Early stopping
    'early_stopping_patience': 2,  # Stop if no improvement for 2 epochs
    
    # Layer freezing strategy (experiment with these)
    'freeze_embeddings': False,  # Freeze embedding layer
    'freeze_encoder_layers': 0,  # Number of encoder layers to freeze (0-12 for base, 0-24 for large)
    
    # Class weighting (for imbalanced datasets)
    'use_class_weights': True,
    
    # Random seed
    'random_state': 42
}

print('Configuration:')
for key, value in CONFIG.items():
    print(f'  {key}: {value}')

print('Configuration:')
for key, value in CONFIG.items():
    print(f'  {key}: {value}')

Word vocab size: 17518
Saved tokenizer to /content/drive/MyDrive/Kaggle/experiments/exp_6_bilstm/tokenizer.json


In [ ]:
# Set random seeds for reproducibility
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['random_state'])

Train shape: (4712, 200)
Test shape: (1178, 200)


In [ ]:
# Custom Dataset class
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx] if self.labels is not None else None
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        
        if label is not None:
            item['labels'] = torch.tensor(label, dtype=torch.long)
        
        return item

In [ ]:
# Model creation with optional layer freezing
def create_model(config):
    model = DebertaV2ForSequenceClassification.from_pretrained(
        config['model_name'],
        num_labels=config['num_labels']
    )
    
    # Freeze embeddings if specified
    if config['freeze_embeddings']:
        for param in model.deberta.embeddings.parameters():
            param.requires_grad = False
        print('Froze embedding layers')
    
    # Freeze encoder layers if specified
    if config['freeze_encoder_layers'] > 0:
        for layer in model.deberta.encoder.layer[:config['freeze_encoder_layers']]:
            for param in layer.parameters():
                param.requires_grad = False
        print(f'Froze first {config["freeze_encoder_layers"]} encoder layers')
    
    # Count trainable parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f'Trainable parameters: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)')
    
    return model

In [ ]:
# Training function with class weights support
def train_epoch(model, dataloader, optimizer, scheduler, device, scaler, config, class_weights=None):
    model.train()
    total_loss = 0
    predictions = []
    true_labels = []
    
    # Create loss function with class weights if provided
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = None
    
    progress_bar = tqdm(dataloader, desc='Training')
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Mixed precision training
        if config['use_fp16']:
            with torch.cuda.amp.autocast():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=None  # Don't use built-in loss
                )
                logits = outputs.logits
                
                # Apply custom loss with class weights if provided
                if criterion is not None:
                    loss = criterion(logits, labels)
                else:
                    # Use default loss
                    outputs_with_loss = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels
                    )
                    loss = outputs_with_loss.loss
                
                loss = loss / config['gradient_accumulation_steps']
            
            scaler.scale(loss).backward()
        else:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=None  # Don't use built-in loss
            )
            logits = outputs.logits
            
            # Apply custom loss with class weights if provided
            if criterion is not None:
                loss = criterion(logits, labels)
            else:
                # Use default loss
                outputs_with_loss = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )
                loss = outputs_with_loss.loss
            
            loss = loss / config['gradient_accumulation_steps']
            loss.backward()
        
        # Gradient accumulation
        if (step + 1) % config['gradient_accumulation_steps'] == 0:
            if config['use_fp16']:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
                optimizer.step()
            
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * config['gradient_accumulation_steps']
        
        # Get predictions
        preds = torch.argmax(logits, dim=1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())
        
        progress_bar.set_postfix({'loss': total_loss / (step + 1)})
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)
    
    return avg_loss, accuracy

In [ ]:
# Evaluation function
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    predictions = []
    probabilities = []
    true_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            total_loss += loss.item()
            
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            
            predictions.extend(preds.cpu().numpy())
            probabilities.extend(probs[:, 1].cpu().numpy())  # Probability of positive class
            true_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    
    # Calculate metrics
    metrics = {
        'loss': avg_loss,
        'accuracy': accuracy_score(true_labels, predictions),
        'precision': precision_score(true_labels, predictions, zero_division=0),
        'recall': recall_score(true_labels, predictions, zero_division=0),
        'f1': f1_score(true_labels, predictions, zero_division=0),
        'roc_auc': roc_auc_score(true_labels, probabilities),
        'log_loss': log_loss(true_labels, probabilities)
    }
    
    return metrics, probabilities

In [ ]:
# Prediction function for test data
def predict(model, dataloader, device):
    model.eval()
    probabilities = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Predicting'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)
            probabilities.extend(probs[:, 1].cpu().numpy())
    
    return np.array(probabilities)

In [ ]:
# Initialize tokenizer
tokenizer = DebertaV2Tokenizer.from_pretrained(CONFIG['model_name'])
print(f'Tokenizer loaded: {CONFIG["model_name"]}')

In [ ]:
# Prepare data
X_text = train['text'].values
y = train['label'].values
X_test_text = test['text'].values

# Calculate class weights for imbalanced data
if CONFIG['use_class_weights']:
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y),
        y=y
    )
    class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
    print(f'Class weights: {class_weights_dict}')
else:
    class_weights_dict = None

In [ ]:
# Cross-validation training loop
skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=CONFIG['random_state'])

fold_metrics = []
oof_predictions = np.zeros(len(y))
test_predictions = []

import time
from datetime import datetime

start_time = time.time()
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

for fold, (train_idx, val_idx) in enumerate(skf.split(X_text, y), 1):
    print(f'\n{"="*60}')
    print(f'FOLD {fold}/{CONFIG["n_folds"]}')
    print(f'{"="*60}')
    
    # Split data
    X_train_fold, X_val_fold = X_text[train_idx], X_text[val_idx]
    y_train_fold, y_val_fold = y[train_idx], y[val_idx]
    
    print(f'Train size: {len(X_train_fold)}, Val size: {len(X_val_fold)}')
    
    # Create datasets
    train_dataset = TextDataset(X_train_fold, y_train_fold, tokenizer, CONFIG['max_length'])
    val_dataset = TextDataset(X_val_fold, y_val_fold, tokenizer, CONFIG['max_length'])
    test_dataset = TextDataset(X_test_text, None, tokenizer, CONFIG['max_length'])
    
    # Create dataloaders (set num_workers=0 for Colab/Jupyter compatibility)
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        num_workers=0,  # Changed from 2 to 0 to avoid multiprocessing issues in Colab
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG['batch_size'] * 2,
        shuffle=False,
        num_workers=0,  # Changed from 2 to 0 to avoid multiprocessing issues in Colab
        pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=CONFIG['batch_size'] * 2,
        shuffle=False,
        num_workers=0,  # Changed from 2 to 0 to avoid multiprocessing issues in Colab
        pin_memory=True
    )
    
    # Create model
    model = create_model(CONFIG)
    model.to(device)
    
    # Prepare class weights tensor if enabled
    class_weights_tensor = None
    if class_weights_dict is not None:
        class_weights_tensor = torch.tensor(
            [class_weights_dict[i] for i in range(CONFIG['num_labels'])],
            dtype=torch.float
        ).to(device)
        print(f'Using class weights: {class_weights_tensor.cpu().numpy()}')
    
    # Optimizer and scheduler
    num_training_steps = len(train_loader) * CONFIG['num_epochs'] // CONFIG['gradient_accumulation_steps']
    num_warmup_steps = int(num_training_steps * CONFIG['warmup_ratio'])
    
    optimizer = AdamW(
        model.parameters(),
        lr=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay']
    )
    
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    
    # Mixed precision scaler
    scaler = torch.cuda.amp.GradScaler() if CONFIG['use_fp16'] else None
    
    print(f'Training steps: {num_training_steps}, Warmup steps: {num_warmup_steps}')
    
    # Training loop with early stopping
    best_val_roc_auc = 0
    patience_counter = 0
    best_model_path = os.path.join(EXPERIMENT_DIR, 'models', f'best_model_fold_{fold}')
    
    for epoch in range(CONFIG['num_epochs']):
        print(f'\nEpoch {epoch + 1}/{CONFIG["num_epochs"]}')
        
        # Train with class weights
        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, scheduler, device, scaler, CONFIG,
            class_weights=class_weights_tensor
        )
        print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
        
        # Validate
        val_metrics, _ = evaluate(model, val_loader, device)
        print(f'Val Loss: {val_metrics["loss"]:.4f}, Val ROC-AUC: {val_metrics["roc_auc"]:.4f}')
        print(f'Val Acc: {val_metrics["accuracy"]:.4f}, Val F1: {val_metrics["f1"]:.4f}')
        
        # Early stopping check
        if val_metrics['roc_auc'] > best_val_roc_auc:
            best_val_roc_auc = val_metrics['roc_auc']
            patience_counter = 0
            # Save best model
            model.save_pretrained(best_model_path)
            tokenizer.save_pretrained(best_model_path)
            print(f'✓ Saved best model (ROC-AUC: {best_val_roc_auc:.4f})')
        else:
            patience_counter += 1
            print(f'No improvement ({patience_counter}/{CONFIG["early_stopping_patience"]})')
            
            if patience_counter >= CONFIG['early_stopping_patience']:
                print('Early stopping triggered!')
                break
    
    # Load best model for final evaluation
    print(f'\nLoading best model from epoch with ROC-AUC: {best_val_roc_auc:.4f}')
    model = DebertaV2ForSequenceClassification.from_pretrained(best_model_path)
    model.to(device)
    
    # Final validation metrics
    val_metrics, val_probs = evaluate(model, val_loader, device)
    oof_predictions[val_idx] = val_probs
    
    # Test predictions
    test_probs = predict(model, test_loader, device)
    test_predictions.append(test_probs)
    
    # Store fold metrics
    fold_metric = {
        'fold': fold,
        'train_size': len(train_idx),
        'val_size': len(val_idx),
        'accuracy': val_metrics['accuracy'],
        'precision': val_metrics['precision'],
        'recall': val_metrics['recall'],
        'f1': val_metrics['f1'],
        'roc_auc': val_metrics['roc_auc'],
        'log_loss': val_metrics['log_loss']
    }
    fold_metrics.append(fold_metric)
    
    print(f'\nFold {fold} Final Metrics:')
    for key, value in fold_metric.items():
        if key not in ['fold', 'train_size', 'val_size']:
            print(f'  {key}: {value:.4f}')
    
    # Clear GPU memory
    del model, optimizer, scheduler, train_loader, val_loader, test_loader
    torch.cuda.empty_cache()

training_time = time.time() - start_time
print(f'\n{"="*60}')
print(f'Training completed in {training_time/60:.2f} minutes')
print(f'{"="*60}')

In [ ]:
# Calculate average metrics across folds
metrics_df = pd.DataFrame(fold_metrics)

average_metrics = {
    'accuracy': metrics_df['accuracy'].mean(),
    'accuracy_std': metrics_df['accuracy'].std(),
    'precision': metrics_df['precision'].mean(),
    'precision_std': metrics_df['precision'].std(),
    'recall': metrics_df['recall'].mean(),
    'recall_std': metrics_df['recall'].std(),
    'f1': metrics_df['f1'].mean(),
    'f1_std': metrics_df['f1'].std(),
    'roc_auc': metrics_df['roc_auc'].mean(),
    'roc_auc_std': metrics_df['roc_auc'].std(),
    'log_loss': metrics_df['log_loss'].mean(),
    'log_loss_std': metrics_df['log_loss'].std()
}

print('\n' + '='*60)
print('AVERAGE METRICS ACROSS ALL FOLDS')
print('='*60)
for key, value in average_metrics.items():
    print(f'{key}: {value:.4f}')

# Identify best fold
best_fold = metrics_df.loc[metrics_df['roc_auc'].idxmax(), 'fold']
print(f'\nBest fold: {best_fold} (ROC-AUC: {metrics_df.loc[metrics_df["roc_auc"].idxmax(), "roc_auc"]:.4f})')

In [ ]:
# Average test predictions across all folds
test_predictions_avg = np.mean(test_predictions, axis=0)

# Create submission file
submission = pd.DataFrame({
    'id': test['id'],
    'probability': test_predictions_avg
})

submission_path = os.path.join(EXPERIMENT_DIR, f'{EXPERIMENT_NAME}_submission.csv')
submission.to_csv(submission_path, index=False)
print(f'\n✓ Submission saved to: {submission_path}')
print(f'Submission shape: {submission.shape}')
print(f'\nFirst few predictions:')
print(submission.head())

In [ ]:
# Save comprehensive metrics JSON (following model tracking guide)
metrics_json = {
    'experiment_name': EXPERIMENT_NAME,
    'model_type': 'DeBERTa-v3',
    'transformer_model': CONFIG['model_name'],
    'timestamp': timestamp,
    'training_time_minutes': training_time / 60,
    
    # Configuration
    'config': CONFIG,
    
    # Cross-validation settings
    'n_folds': CONFIG['n_folds'],
    'best_fold': int(best_fold),
    
    # Average metrics
    'average_metrics': average_metrics,
    
    # Per-fold metrics
    'fold_metrics': fold_metrics,
    
    # Model paths
    'fold_models': [
        {
            'fold': i,
            'model_path': f'models/best_model_fold_{i}',
            'metrics': fold_metrics[i-1]
        }
        for i in range(1, CONFIG['n_folds'] + 1)
    ],
    
    # Additional info
    'total_train_samples': len(y),
    'total_test_samples': len(X_test_text),
    'class_distribution': {
        'class_0': int(np.sum(y == 0)),
        'class_1': int(np.sum(y == 1))
    }
}

metrics_path = os.path.join(EXPERIMENT_DIR, 'metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics_json, f, indent=2)

print(f'✓ Metrics saved to: {metrics_path}')

In [ ]:
# Display final summary
print('\n' + '='*60)
print('EXPERIMENT SUMMARY')
print('='*60)
print(f'Experiment: {EXPERIMENT_NAME}')
print(f'Model: {CONFIG["model_name"]}')
print(f'Training Time: {training_time/60:.2f} minutes')
print(f'\nBest Fold: {best_fold}')
print(f'Average ROC-AUC: {average_metrics["roc_auc"]:.4f} ± {average_metrics["roc_auc_std"]:.4f}')
print(f'Average F1: {average_metrics["f1"]:.4f} ± {average_metrics["f1_std"]:.4f}')
print(f'Average Accuracy: {average_metrics["accuracy"]:.4f} ± {average_metrics["accuracy_std"]:.4f}')
print(f'\nArtifacts saved to: {EXPERIMENT_DIR}')
print('  ✓ Model files (per fold)')
print('  ✓ metrics.json')
print('  ✓ submission CSV')
print('='*60)

## Experimentation Ideas

If you want to further improve the model, try adjusting these settings in the CONFIG cell:

### 1. Layer Freezing Strategies
- **No freezing** (current): Train all layers (slower, more flexible)
- **Freeze embeddings**: `freeze_embeddings=True` (slightly faster)
- **Freeze lower layers**: `freeze_encoder_layers=6` (freeze first 6 layers, train upper 6)
- **Freeze most layers**: `freeze_encoder_layers=10` (only train last 2 encoder layers + classifier)

### 2. Model Sizes
- **deberta-v3-small**: 44M parameters, fastest training, good for limited resources
- **deberta-v3-base** (current): 184M parameters, balanced performance/speed
- **deberta-v3-large**: 434M parameters, best performance but needs more GPU memory
  - If using deberta-v3-large, reduce batch_size to 8 or 4
- **deberta-v3-xsmall**: 22M parameters, ultra-fast for prototyping

### 3. Hyperparameter Tuning
- **Learning rate**: Try 1e-5, 2e-5 (current), 3e-5, 5e-5
- **Batch size**: Adjust based on GPU memory (8, 16, 32)
  - Note: DeBERTa may require smaller batch sizes than RoBERTa due to larger model size
- **Max length**: 128, 256 (current), 512 (longer = more context but slower)
- **Epochs**: 3-5 epochs typical for transformers
- **Warmup ratio**: 0.0, 0.1 (current), 0.2

### 4. Training Techniques
- **Gradient accumulation**: Increase `gradient_accumulation_steps` for larger effective batch size
- **Scheduler**: Try `get_cosine_schedule_with_warmup` instead of linear
- **Dropout**: Add custom dropout layers in classifier head
- **Layer-wise learning rate decay**: Use different learning rates for different layers

### 5. Data Strategies
- **Text augmentation**: Back-translation, synonym replacement
- **Longer training**: More epochs with early stopping
- **Test-time augmentation**: Multiple predictions with different max_length or dropout

### 6. DeBERTa-Specific Advantages
- **Disentangled attention mechanism**: Better handling of position and content
- **Enhanced mask decoder**: Improved performance on downstream tasks
- **Generally outperforms**: RoBERTa and BERT on many benchmarks with similar or smaller size